# 每月调仓决策助手

**策略**: U11 + residual_mom + n=4 + rank weighting

**操作流程**（每月第一个交易日做一次）：

1. 在下面的 Inputs cell 里填写**当前持仓**（IBKR 账户里实际持有的 ETF 和股数）+ **可用现金余额**
2. 点 Cell → Run All（或 ⇧⏎ 一格一格执行）
3. 最后一个 cell 输出**交易清单**：买卖什么、多少股
4. 去 IBKR 下单（市价单 OK，monthly rebalance 不在乎几个 bp 的滑点）

**信号源**：yfinance 实时数据（免费）。  
**理论依据**：见 `docs/research_plan.md` 和 `reports/strategy_backtest_report.md`。  
**Backtest 表现** (2010-2026, 16 年, U11+rank+n=4): CAGR 14.3%, Sharpe 1.20, MDD -15.6%。

## 1. Imports + 配置（不要改）

In [8]:
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# Strategy parameters (locked in after diagnostic tests)
# IEFA 替代 EFA (相同发达国际市场，expense ratio 0.07% vs 0.32%)
# GLDM 替代 GLD (相同实物黄金，expense ratio 0.10% vs 0.40%)
UNIVERSE = ['SPY', 'QQQ', 'IEFA', 'TLT', 'GLDM',
            'VWO', 'HYG', 'VNQ', 'IEF', 'DBC', 'SOXX']
LONG_N   = 4                  # 持有数量
SCHEME   = 'rank'             # 权重方案: rank = 40/30/20/10

# Rank weights for n=4 (top → bottom): 40% / 30% / 20% / 10%
RANK_WEIGHTS_N4 = [4/10, 3/10, 2/10, 1/10]

print(f'Strategy: U{len(UNIVERSE)} + residual_mom + n={LONG_N} + {SCHEME} weighting')
print(f'Universe: {UNIVERSE}')

Strategy: U11 + residual_mom + n=4 + rank weighting
Universe: ['SPY', 'QQQ', 'IEFA', 'TLT', 'GLDM', 'VWO', 'HYG', 'VNQ', 'IEF', 'DBC', 'SOXX']


## 2. ★ Inputs ★（**每月只改这一格**）

把 IBKR 账户里**今天**的实际状态填进去：

In [9]:
# 当前持仓：ticker -> 持有股数（整数）。
# 没持有的 ETF 不要列；首次运行就留空 dict {}
current_holdings = {
    # 'SPY': 12,
    # 'TLT': 25,
    # 'GLD': 18,
    # 'SOXX': 5,
}

# 当前可用现金余额（USD）。包含本月刚 deposit 进去的 $3k。
cash_balance = 23000.0     # 例：$20k 起始 + $3k 月度 deposit

# As-of date：通常设为今天。可以指定历史日期做回溯演练。
# 留 None 即用 today。
as_of_date = None          # 例如: '2026-06-02'

# ─── 不需要改的派生变量 ───
if as_of_date is None:
    as_of_date = pd.Timestamp.today().normalize()
else:
    as_of_date = pd.Timestamp(as_of_date)
print(f'As of: {as_of_date.date()}')
print(f'Cash:  ${cash_balance:,.2f}')
print(f'Current holdings: {dict(current_holdings) or "(empty)"}')

As of: 2026-05-16
Cash:  $23,000.00
Current holdings: (empty)


## 3. 拉数据（自动）

In [10]:
# Need 252+30 trading days of history for the 12-1 momentum + beta regression.
# Fetch 400 calendar days to be safe.
fetch_start = (as_of_date - timedelta(days=400)).strftime('%Y-%m-%d')
fetch_end   = (as_of_date + timedelta(days=1)).strftime('%Y-%m-%d')

raw = yf.download(UNIVERSE, start=fetch_start, end=fetch_end,
                  progress=False, auto_adjust=True)
closes = raw['Close'].dropna(how='all')

print(f'Data fetched: {len(closes)} trading days from '
      f'{closes.index.min().date()} to {closes.index.max().date()}')

# Latest closes (used for converting target $ -> target shares)
latest_prices = closes.iloc[-1]
print(f'\nLatest prices (as of {closes.index[-1].date()}):')
for t in UNIVERSE:
    p = latest_prices.get(t)
    print(f'  {t:>4}: ${p:>8.2f}' if pd.notna(p) else f'  {t:>4}: MISSING')

Data fetched: 275 trading days from 2025-04-11 to 2026-05-15

Latest prices (as of 2026-05-15):
   SPY: $  739.17
   QQQ: $  708.93
  IEFA: $   95.21
   TLT: $   83.66
  GLDM: $   89.98
   VWO: $   58.44
   HYG: $   79.46
   VNQ: $   93.91
   IEF: $   93.51
   DBC: $   31.19
  SOXX: $  508.52


## 4. 计算信号（residual_mom）

对每个 ETF：剥除其相对 SPY 的 beta，残差累积回报作为信号。**这就是 `3d_xsmom_lo` 里 `signal=residual_mom` 的逻辑**。

In [11]:
def residual_mom_latest(asset_px: pd.Series, market_px: pd.Series) -> float:
    """Compute residual_mom for the latest date.
    Uses prices from t-252 to t-21 (231-day window ending 1 month ago)."""
    asset_ret = asset_px.pct_change()
    mkt_ret = market_px.pct_change()
    aligned = pd.concat([asset_ret, mkt_ret], axis=1, keys=['a', 'm']).dropna()
    if len(aligned) < 252:
        return np.nan
    win = aligned.iloc[-252:-21]
    if len(win) < 100:
        return np.nan
    a, m = win['a'].values, win['m'].values
    cov = float(np.cov(a, m, ddof=0)[0, 1])
    var = float(np.var(m, ddof=0))
    beta = cov / var if var > 1e-12 else 1.0
    return float((a - beta * m).sum())

signals = {}
for t in UNIVERSE:
    if t not in closes.columns or closes[t].isna().all():
        print(f'  WARN: {t} no data, skipping')
        continue
    s = residual_mom_latest(closes[t], closes['SPY'])
    signals[t] = s

sig_df = pd.DataFrame({
    'signal':     [signals[t] for t in UNIVERSE],
    'rank':       0,  # filled below
    'positive':   [signals[t] > 0 for t in UNIVERSE],
    'last_price': [latest_prices.get(t) for t in UNIVERSE],
}, index=UNIVERSE)
sig_df = sig_df.sort_values('signal', ascending=False)
sig_df['rank'] = range(1, len(sig_df) + 1)

print(f'\nSignal table (sorted descending):')
with pd.option_context('display.float_format', '{:+.4f}'.format):
    print(sig_df.to_string())


Signal table (sorted descending):
      signal  rank  positive  last_price
DBC  +0.3931     1      True    +31.1900
GLDM +0.3888     2      True    +89.9800
SOXX +0.2989     3      True   +508.5200
VWO  +0.0558     4      True    +58.4400
IEFA +0.0530     5      True    +95.2100
IEF  +0.0490     6      True    +93.5100
TLT  +0.0346     7      True    +83.6600
VNQ  +0.0309     8      True    +93.9100
HYG  +0.0214     9      True    +79.4600
SPY  +0.0000    10      True   +739.1700
QQQ  -0.0157    11     False   +708.9300


## 5. 选 top 4 + 分配仓位

In [12]:
# 只取正动量 top 4
positives = sig_df[sig_df['positive']].head(LONG_N)
k = len(positives)

if k == 0:
    print('⚠️  没有正动量 ETF — 全部持现金。本月不需要交易（如果当前已经是现金）。')
    targets_pct = {}
elif k < LONG_N:
    print(f'⚠️  只有 {k}/{LONG_N} 个 ETF 正动量。会有 {(1-k/LONG_N)*100:.0f}% 仓位留作现金。')
    # rank weights scaled to deployed = k/LONG_N
    weights_top4 = [(LONG_N - i) for i in range(LONG_N)]   # [4, 3, 2, 1]
    raw = weights_top4[:k]   # 取前 k 个
    sum_all = sum(weights_top4)
    targets_pct = {positives.index[i]: raw[i] / sum_all for i in range(k)}
else:
    # 完整 4 个 winners，40/30/20/10
    targets_pct = {positives.index[i]: RANK_WEIGHTS_N4[i] for i in range(LONG_N)}

# 计算总组合价值（现金 + 现有持仓市值）
holdings_value = sum(shares * latest_prices.get(t, 0)
                     for t, shares in current_holdings.items()
                     if t in latest_prices and pd.notna(latest_prices.get(t)))
total_value = cash_balance + holdings_value
print(f'\n当前组合：')
print(f'  持仓市值 ${holdings_value:,.2f} + 现金 ${cash_balance:,.2f} = 总值 ${total_value:,.2f}')

print(f'\n目标分配（rank weighting）：')
for t, w in targets_pct.items():
    target_dollars = total_value * w
    target_shares  = int(round(target_dollars / latest_prices[t]))
    print(f'  {t:>4}  rank #{positives.index.get_loc(t)+1}  '
          f'weight {w*100:>5.1f}%  → ${target_dollars:>10,.0f} ≈ {target_shares} shares @ ${latest_prices[t]:.2f}')


当前组合：
  持仓市值 $0.00 + 现金 $23,000.00 = 总值 $23,000.00

目标分配（rank weighting）：
   DBC  rank #1  weight  40.0%  → $     9,200 ≈ 295 shares @ $31.19
  GLDM  rank #2  weight  30.0%  → $     6,900 ≈ 77 shares @ $89.98
  SOXX  rank #3  weight  20.0%  → $     4,600 ≈ 9 shares @ $508.52
   VWO  rank #4  weight  10.0%  → $     2,300 ≈ 39 shares @ $58.44


## 6. 交易清单（**复制到 IBKR 执行**）

In [13]:
# 计算 target shares per ticker
target_shares = {}
for t in UNIVERSE:
    if t in targets_pct:
        target_shares[t] = int(round(total_value * targets_pct[t] / latest_prices[t]))
    else:
        target_shares[t] = 0

# Diff: target - current
trade_rows = []
for t in UNIVERSE:
    curr = current_holdings.get(t, 0)
    tgt = target_shares.get(t, 0)
    diff = tgt - curr
    if diff == 0 and curr == 0:
        continue  # nothing to show
    if pd.isna(latest_prices.get(t)):
        continue
    action = 'HOLD' if diff == 0 else ('BUY' if diff > 0 else 'SELL')
    trade_rows.append({
        'action':  action,
        'ticker':  t,
        'current': curr,
        'target':  tgt,
        'delta':   diff,
        'price':   latest_prices[t],
        'amount':  abs(diff) * latest_prices[t],
    })

trade_df = pd.DataFrame(trade_rows)
if trade_df.empty:
    print('✅ 当前持仓已与目标一致，本月无需交易。')
else:
    trade_df = trade_df.sort_values(['action', 'amount'], ascending=[True, False])
    # Print in execution order: SELLs first (free up cash), then BUYs
    sells = trade_df[trade_df.action == 'SELL']
    buys  = trade_df[trade_df.action == 'BUY']
    holds = trade_df[trade_df.action == 'HOLD']

    print('═' * 60)
    print(f'   交易清单（{as_of_date.date()}）')
    print('═' * 60)
    if not sells.empty:
        print('\n→ 先 SELL（先卖出腾现金）:')
        for _, r in sells.iterrows():
            print(f"   SELL  {r['ticker']:>4}  {r['delta']:>+5d} shares  @ ~${r['price']:.2f}  = ${r['amount']:>10,.0f}")
    if not buys.empty:
        print('\n→ 再 BUY:')
        for _, r in buys.iterrows():
            print(f"   BUY   {r['ticker']:>4}  {r['delta']:>+5d} shares  @ ~${r['price']:.2f}  = ${r['amount']:>10,.0f}")
    if not holds.empty:
        print('\n→ HOLD（不动）:')
        for _, r in holds.iterrows():
            print(f"   HOLD  {r['ticker']:>4}  {r['current']:>5d} shares  (no change)")

    total_sell = sells['amount'].sum() if not sells.empty else 0
    total_buy  = buys['amount'].sum()  if not buys.empty  else 0
    print(f"\n💵 净现金变动: SELL ${total_sell:,.0f} - BUY ${total_buy:,.0f} "
          f"= ${total_sell - total_buy:+,.0f}")
    print(f"💰 交易完成后预计现金余额: ${cash_balance + total_sell - total_buy:,.0f}")
    print('═' * 60)

════════════════════════════════════════════════════════════
   交易清单（2026-05-16）
════════════════════════════════════════════════════════════

→ 再 BUY:
   BUY    DBC   +295 shares  @ ~$31.19  = $     9,201
   BUY   GLDM    +77 shares  @ ~$89.98  = $     6,928
   BUY   SOXX     +9 shares  @ ~$508.52  = $     4,577
   BUY    VWO    +39 shares  @ ~$58.44  = $     2,279

💵 净现金变动: SELL $0 - BUY $22,985 = $-22,985
💰 交易完成后预计现金余额: $15
════════════════════════════════════════════════════════════


## 7. 预期指标（backtest baseline，每月对照参考）

下面是 QC backtest 2010-2026 的预期范围。**真正比较要 12+ 个月之后才有意义**——短期内 noise 远大于 signal。

In [ ]:
# QC backtest 2010-2026 (16.4 年, U11 + residual_mom + n=4 + rank)
# 实测数字（含 commission + 5bp 滑点）
BT_CAGR        = 0.1363    # 13.63%
BT_SHARPE      = 0.778
BT_MDD         = -0.176    # -17.6%
BT_ANN_VOL     = 0.098     # 9.8% annualized
BT_MONTHLY_MU  = (1 + BT_CAGR) ** (1/12) - 1  # geometric monthly
BT_MONTHLY_SD  = BT_ANN_VOL / (12 ** 0.5)

# Calibrated forward expectation (Sharpe haircut to 0.5-0.7)
FWD_CAGR_LO    = 0.09      # 9%
FWD_CAGR_HI    = 0.12      # 12%
FWD_SHARPE_LO  = 0.5
FWD_SHARPE_HI  = 0.7
FWD_MDD_LO     = -0.28     # -28%
FWD_MDD_HI     = -0.20     # -20%

print('=== Backtest baseline (QC cloud 2010-2026) ===')
print(f'  CAGR:        {BT_CAGR:+.2%}')
print(f'  Sharpe:      {BT_SHARPE:+.3f}')
print(f'  MDD:         {BT_MDD:+.2%}')
print(f'  Monthly μ:   {BT_MONTHLY_MU:+.2%}')
print(f'  Monthly σ:   {BT_MONTHLY_SD:.2%}')
print()
print('=== Calibrated forward expectation (打折后) ===')
print(f'  CAGR range:    {FWD_CAGR_LO:+.0%} to {FWD_CAGR_HI:+.0%}')
print(f'  Sharpe range:  {FWD_SHARPE_LO:.2f} to {FWD_SHARPE_HI:.2f}')
print(f'  MDD expected:  {FWD_MDD_LO:+.0%} to {FWD_MDD_HI:+.0%}')
print()
print('=== This month预期单月收益 ===')
print(f'  Mean:     {BT_MONTHLY_MU:+.2%}')
print(f'  95% CI:   [{BT_MONTHLY_MU - 1.96*BT_MONTHLY_SD:+.2%}, {BT_MONTHLY_MU + 1.96*BT_MONTHLY_SD:+.2%}]')
print(f'  68% CI:   [{BT_MONTHLY_MU - BT_MONTHLY_SD:+.2%}, {BT_MONTHLY_MU + BT_MONTHLY_SD:+.2%}]')
print()
print('注意：单月 σ 是 2.8%，意味着 ±5.5% 都在正常范围内。')
print('单月大涨大跌不要焦虑——只在累积 6+ 个月偏离预期时再 review。')

## 8. 实盘 vs 预期跟踪

自动从 `rebalance_log_*.txt` 抓你历次记录的资产总值，对比 backtest 预期。**至少跑 3-6 个月才有 informative 比较**。

实操：每次跑完 notebook 都会生成 `rebalance_log_YYYY-MM-DD.txt`，自动包含 `Total portfolio value`。这个 cell 自动收集，不需要手动维护表。

In [ ]:
import glob
import re
from pathlib import Path

# Auto-parse all historical rebalance logs in current dir
log_paths = sorted(glob.glob('rebalance_log_*.txt'))
realized = {}
for log_path in log_paths:
    date_str = Path(log_path).stem.replace('rebalance_log_', '')
    try:
        with open(log_path) as f:
            text = f.read()
        m = re.search(r'Total portfolio value:\s*\$([\d,]+\.\d+)', text)
        if m:
            realized[date_str] = float(m.group(1).replace(',', ''))
    except Exception:
        continue

# Manual overrides / supplements (if needed)
realized.update({
    # '2026-05-16': 20000,   # initial entry (manual)
    # add manual entries here if log file missing
})

if len(realized) < 2:
    print(f'⚠ 只有 {len(realized)} 条 log，需要至少 2 个月才能算 return。')
    print('  跑过几次月度调仓后再回来看这格。')
    print(f'  已有 logs: {list(realized.keys())}')
else:
    s = pd.Series(realized).sort_index()
    s.index = pd.to_datetime(s.index)
    rets = s.pct_change().dropna()
    months = len(rets)

    cum_actual = s.iloc[-1] / s.iloc[0] - 1
    expected_cum_mu = (1 + BT_MONTHLY_MU) ** months - 1
    expected_cum_sd = BT_MONTHLY_SD * (months ** 0.5)
    expected_lo_95 = expected_cum_mu - 1.96 * expected_cum_sd
    expected_hi_95 = expected_cum_mu + 1.96 * expected_cum_sd

    print(f'=== 跟踪期 {s.index[0].date()} 至 {s.index[-1].date()} ({months} 个月度间隔) ===')
    print(f'起始资金:    ${s.iloc[0]:>10,.0f}')
    print(f'当前资金:    ${s.iloc[-1]:>10,.0f}')
    print()
    print(f'累积实盘 return:  {cum_actual:+.2%}')
    print(f'累积预期 mean:    {expected_cum_mu:+.2%}')
    print(f'累积预期 95% CI:  [{expected_lo_95:+.2%}, {expected_hi_95:+.2%}]')
    print()

    # Verdict
    if expected_lo_95 <= cum_actual <= expected_hi_95:
        print(f'✓ 在预期范围内 (normal)')
    elif cum_actual > expected_hi_95:
        print(f'★ 跑赢预期 (above 95% CI)')
    else:
        print(f'✗ 低于预期 95% CI 下界——观察是否持续多个月再下结论')

    print()
    print('月度 returns:')
    for d, r in rets.items():
        in_normal = abs(r - BT_MONTHLY_MU) <= 1.96 * BT_MONTHLY_SD
        flag = ' ' if in_normal else '!'
        print(f'  {d.strftime("%Y-%m"):s} {r:+8.2%} {flag}')

    # Realized annualized stats (only if 6+ months)
    if months >= 6:
        realized_mean = rets.mean() * 12
        realized_sd = rets.std() * (12 ** 0.5)
        realized_sharpe = realized_mean / realized_sd if realized_sd > 0 else 0
        print()
        print(f'=== 实盘 annualized (最近 {months} 个月) ===')
        print(f'  Return:  {realized_mean:+.2%}  (backtest {BT_CAGR:+.2%})')
        print(f'  Vol:     {realized_sd:.2%}  (backtest {BT_ANN_VOL:.2%})')
        print(f'  Sharpe:  {realized_sharpe:+.3f}  (backtest {BT_SHARPE:+.3f})')
        if realized_sharpe < FWD_SHARPE_LO - 0.2 and months >= 12:
            print(f'  ⚠ Sharpe 显著低于 forward 下界，建议 review')

## 9. 健康检查 + 备忘

执行 IBKR 下单时：
- 用 **market order**，monthly rebalance 不在乎 1-2 bp 滑点
- 先卖后买（先腾出现金）
- 整数股，零碎现金留作下月 buffer
- 执行完后**记录当时的持仓数字**，下个月填进 Inputs cell

**异常情况**：
- 如果某月 signal 全部 ≤ 0（全资产负动量）→ 全部清仓持现金。这是策略的避险机制，不要质疑。
- 如果连续 3+ 个月持仓完全一致 → 信号稳定，正常。
- 如果某月 turnover > 60%（一半以上仓位换掉）→ 检查是不是 signal 计算异常。

**调仓时间**：每月**第一个交易日** 10:00 ET 后跑 notebook（确保前一日数据已 settle）。如果错过几天问题不大。

**Deposit 节奏**：下次 $3k 到账后，更新 `cash_balance`，下月调仓自动会把新现金 deploy 进去。

In [14]:
# 把本次决策结果保存到文本文件（可选，便于事后审计）
import os
log_path = f'rebalance_log_{as_of_date.date()}.txt'
log_full_path = os.path.join(os.path.dirname(os.path.abspath('__file__')) if False else '.', log_path)

with open(log_path, 'w') as f:
    f.write(f'Rebalance decision @ {as_of_date.date()}\n')
    f.write(f'Strategy: U{len(UNIVERSE)} + residual_mom + n={LONG_N} + {SCHEME}\n')
    f.write(f'\nSignal table:\n{sig_df.to_string()}\n')
    f.write(f'\nCurrent holdings: {current_holdings}\n')
    f.write(f'Cash balance: ${cash_balance:,.2f}\n')
    f.write(f'Total portfolio value: ${total_value:,.2f}\n')
    f.write(f'\nTarget weights:\n')
    for t, w in targets_pct.items():
        f.write(f'  {t}: {w*100:.1f}%  ({target_shares[t]} shares)\n')
    if not trade_df.empty:
        f.write(f'\nTrades:\n{trade_df.to_string()}\n')

print(f'✅ 决策记录已保存到 {log_path}')

✅ 决策记录已保存到 rebalance_log_2026-05-16.txt
